# SymboLM (v1.2) — Stage 3 Concept GRPO Training (Dual Tesla T4)

**Author & Principal Architect:** Sanish Gyawali  
**AI Systems Collaborator:** Antigravity (Google DeepMind)  
**Hardware:** Kaggle Dual NVIDIA Tesla T4 (2x 16 GB = 32 GB VRAM)  
**Repository:** [https://github.com/gyawalisanish0/symboLM](https://github.com/gyawalisanish0/symboLM)  

---  
### Core Objectives:
1. Train policy on **Register v2.0 (Hierarchical Cognitive Architecture)** with macro-concept planning (`<reg:plan>`), micro-symbolic AST execution (`<reg:deduce>`), and verification (`<reg:verify>`).
2. Eliminate the Binary Trap (1-token guessing vs verbose CoT fallback) using anti-guessing difficulty penalties ($-0.80$).
3. Reward domain invariant identification across 7 Olympiad mathematical families ($+0.45$).
4. Retain 100% of older learned capabilities via Experience Replay (1,600 unified samples).

In [ ]:
# Cell 1: Environment & Hardware Verification
import os
import sys
import torch

print('Python version:', sys.version)
print('Torch version :', torch.__version__)
if torch.cuda.is_available():
    print(f'[OK] NVIDIA CUDA detected! Found {torch.cuda.device_count()} GPU(s):')
    for i in range(torch.cuda.device_count()):
        mem = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)} ({mem:.1f} GB VRAM)')
else:
    print('[Warning] No CUDA GPU found!')


In [ ]:
# Cell 2: Install required libraries & resolve package conflicts
!pip install -q --upgrade pip
!pip uninstall -y -q torchao
!pip install -q transformers peft trl accelerate sympy datasets bitsandbytes


In [ ]:
# Cell 3: Clone latest SymboLM repository (Commit with Register v2.0 & Concept Curriculum)
!rm -rf /kaggle/working/symboLM
!git clone https://github.com/gyawalisanish0/symboLM.git /kaggle/working/symboLM
%cd /kaggle/working/symboLM


In [ ]:
# Cell 4: Mount and unpack Stage 2 GRPO Policy Adapter as warm-start base
import os
import shutil
from pathlib import Path

target_dir = Path('/kaggle/working/symboLM/checkpoints/stage2_adapter')
target_dir.mkdir(parents=True, exist_ok=True)

found = False
# 1. Search for Stage 2 archive in input sources
for arc in Path('/kaggle/input').glob('**/*grpo*.tar.gz'):
    print(f'[OK] Found Stage 2 archive at: {arc}. Unpacking...')
    os.system(f'tar -xzf "{arc}" -C "{target_dir}"')
    found = True
    break

# 2. Search for extracted adapter_config.json
if not found:
    for p in Path('/kaggle/input').glob('**/grpo_adapter'):
        if (p / 'adapter_config.json').exists():
            print(f'[OK] Found extracted GRPO adapter directory at: {p}')
            for item in p.iterdir():
                if item.is_file():
                    shutil.copy2(item, target_dir / item.name)
            found = True
            break

# 3. Fallback: Check Stage 1 SFT if Stage 2 not mounted
if not found:
    for arc in Path('/kaggle/input').glob('**/*sft*.tar.gz'):
        print(f'[Notice] Stage 2 not found, using SFT archive: {arc}')
        os.system(f'tar -xzf "{arc}" -C "{target_dir}"')
        found = True
        break

print('[Status] Target Warm-Start Adapter Contents:')
for f in sorted(target_dir.iterdir()):
    sz = f.stat().st_size / 1e6 if f.is_file() else 0
    print(f'  {f.name} ({sz:.2f} MB)' if f.is_file() else f'  {f.name}/')


In [ ]:
# Cell 5: Generate the 1,600-sample Unified Stage 3 Curriculum (Concept Math + Anchors)
!python -m data.build_concept_math_dataset --count 1400 --output ./data/concept_math_curriculum.jsonl
!python -m data.build_stage3_curriculum


In [ ]:
# Cell 6: Launch Stage 3 Concept GRPO Training on Dual Tesla T4
!CUDA_VISIBLE_DEVICES=0 python -m training.grpo_train \
    --base_model deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B \
    --sft_adapter ./checkpoints/stage2_adapter \
    --output_dir ./checkpoints/stage3_adapter \
    --data_dir ./data \
    --num_generations 4 \
    --learning_rate 0.000003 \
    --max_steps 250


In [ ]:
# Cell 7: Post-Training Policy Verification (Macro-Plan Emission & Anti-Guess Audit)
print('[AUDIT 1] Invariant Algebra (Polynomial Symmetry):')
!CUDA_VISIBLE_DEVICES=0 python -m inference.generate \
    --base_model deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B \
    --adapter_path ./checkpoints/stage3_adapter \
    --prompt "If x + 1/x = 5, find the exact value of x^4 + 1/x^4."

print('\n[AUDIT 2] Geometric Invariant (Right Triangle Inradius):')
!CUDA_VISIBLE_DEVICES=0 python -m inference.generate \
    --base_model deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B \
    --adapter_path ./checkpoints/stage3_adapter \
    --prompt "In a right triangle with inradius r = 6 and perimeter P = 72, find the area of the triangle."

print('\n[AUDIT 3] Zero-Shot Factual Recall (Bypass Verification):')
!CUDA_VISIBLE_DEVICES=0 python -m inference.generate \
    --base_model deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B \
    --adapter_path ./checkpoints/stage3_adapter \
    --prompt "What is the atomic number of Carbon?"


In [ ]:
# Cell 8: Package and archive final Stage 3 policy checkpoints
!tar -czf /kaggle/working/symboLM_stage3_final_policy.tar.gz -C /kaggle/working/symboLM/checkpoints/stage3_adapter .
print("[COMPLETE] Stage 3 Concept GRPO complete! Policy weights saved to /kaggle/working/symboLM_stage3_final_policy.tar.gz")
